# Credit Risk Model Inference & Evaluation Demo
This notebook demonstrates how to load the production model artifact and run inference with SHAP explanations.

In [ ]:
import os
import joblib
import pandas as pd
import numpy as np

# Load artifact
model_path = os.path.join("model", "loan_model.pkl")
artifact = joblib.load(model_path)
preprocessor = artifact["preprocessor"]
calibrated_model = artifact["calibrated_model"]
explainer = artifact["explainer"]
feature_names = artifact["feature_names"]

print(f"Model loaded successfully with {len(feature_names)} features.")

In [ ]:
# Sample applicant data
sample_applicant = pd.DataFrame([{
    "person_age": 29,
    "person_income": 70000.0,
    "person_emp_length": 5.0,
    "loan_amnt": 10000.0,
    "loan_int_rate": 8.5,
    "loan_percent_income": 0.14,
    "cb_person_cred_hist_length": 6,
    "person_home_ownership": "MORTGAGE",
    "loan_intent": "PERSONAL",
    "loan_grade": "A",
    "cb_person_default_on_file": "N"
}])

transformed = preprocessor.transform(sample_applicant)
pred = calibrated_model.predict(transformed)[0]
prob = calibrated_model.predict_proba(transformed)[0][1]

print(f"Approval Decision: {'Approved' if pred == 0 else 'Rejected'}")
print(f"Calibrated Default Probability: {prob * 100:.2f}%")

In [ ]:
shap_values = explainer.shap_values(transformed)
raw_vals = shap_values[1] if isinstance(shap_values, list) else shap_values
vals = np.array(raw_vals).reshape(-1)

shap_df = pd.DataFrame({
    "feature": feature_names,
    "shap_value": vals
}).sort_values(by="shap_value", ascending=False)

shap_df.head(10)